# Live Mistral PDF Smoke Test

This notebook tests the thing that matters: **can Mistral OCR process a Gazette PDF URL through this package and produce usable bundle files?**

It makes a live Mistral API call against a public Kenya Gazette PDF URL, caches the raw Mistral OCR JSON, runs the package parser, validates the envelope, writes bundles, and previews the output.

You need `MISTRAL_API_KEY` in the notebook kernel environment or in the repo `.env` file. This is a real network/API test and may be billable.


## What You Are Testing

Run the cells top to bottom. A successful run proves:

1. The notebook kernel can import this package.
2. The kernel has access to `MISTRAL_API_KEY`.
3. Mistral accepts the configured public PDF URL and returns OCR JSON.
4. The package caches that raw OCR JSON under `examples/_live_mistral_pdf_test/stage`.
5. The package converts OCR markdown into an envelope and validates it.
6. The bundle writer creates JSON and markdown outputs under `examples/_live_mistral_pdf_test/bundles`.

This notebook does **not** test local PDF upload. The current package supports live Mistral OCR for PDF URLs only; local PDF files currently work only in replay mode.

In [2]:
from pathlib import Path
import os
import sys

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

cwd = Path.cwd()
repo_root = next(
    (
        candidate
        for candidate in (cwd, cwd.parent)
        if (candidate / "gazette_mistral_pipeline" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from gazette_mistral_pipeline import (
        Bundles,
        GazetteConfig,
        parse_url,
        validate_envelope_json,
        write_envelope,
    )
except ModuleNotFoundError as exc:
    if exc.name == "pydantic" and repo_root is not None:
        raise ModuleNotFoundError(
            "This notebook kernel is missing package dependencies. "
            "Run this in a notebook cell, then restart the kernel: "
            f'%pip install -e "{repo_root}"'
        ) from exc
    raise

base_dir = repo_root or cwd


def show_markdown(text: str) -> None:
    # Always print plain text so Cursor/Jupyter cannot hide the important result.
    print(text)
    if Markdown is not None and display is not None:
        display(Markdown(text))


def display_path(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(base_dir.resolve())).replace("\\", "/")
    except ValueError:
        return str(path)


def load_env_file(path: Path) -> bool:
    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
    return True


dotenv_loaded = load_env_file(base_dir / ".env")
api_key_ready = bool(os.environ.get("MISTRAL_API_KEY", "").strip())

show_markdown(
    "## Setup Ready\n\n"
    f"- Package import: `ok`\n"
    f"- Repo root: `{display_path(base_dir)}`\n"
    f"- `.env` loaded: `{dotenv_loaded}`\n"
    f"- `MISTRAL_API_KEY` available: `{api_key_ready}`"
)


## Setup Ready

- Package import: `ok`
- Repo root: `.`
- `.env` loaded: `True`
- `MISTRAL_API_KEY` available: `True`


## Setup Ready

- Package import: `ok`
- Repo root: `.`
- `.env` loaded: `True`
- `MISTRAL_API_KEY` available: `True`

## Live PDF Configuration

This cell chooses the public PDF URL and output folders for the live Mistral run. Change `PDF_URL` if you want to test a different public PDF URL.


In [6]:
PDF_URL = "https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf"

examples_dir = base_dir / "examples"
output_root = examples_dir / "_live_mistral_pdf_test"
stage_dir = output_root / "stage"
bundle_dir = output_root / "bundles"

live_config = GazetteConfig(
    runtime={
        "allow_live_mistral": True,
        "output_dir": stage_dir,
    }
)

show_markdown(
    "## Live Test Configuration\n\n"
    f"- PDF URL: `{PDF_URL}`\n"
    f"- Mistral model: `{live_config.mistral.model}`\n"
    f"- API key env var: `{live_config.mistral.api_key_env}`\n"
    f"- Stage directory: `{display_path(stage_dir)}`\n"
    f"- Bundle directory: `{display_path(bundle_dir)}`"
)


## Live Test Configuration

- PDF URL: `https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf`
- Mistral model: `mistral-ocr-latest`
- API key env var: `MISTRAL_API_KEY`
- Stage directory: `examples/_live_mistral_pdf_test/stage`
- Bundle directory: `examples/_live_mistral_pdf_test/bundles`


## Live Test Configuration

- PDF URL: `https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf`
- Mistral model: `mistral-ocr-latest`
- API key env var: `MISTRAL_API_KEY`
- Stage directory: `examples/_live_mistral_pdf_test/stage`
- Bundle directory: `examples/_live_mistral_pdf_test/bundles`

## Run Live Mistral OCR On The PDF

This is the real test. It calls Mistral OCR using `document_url`, caches the raw OCR JSON, validates the parsed envelope, and writes the selected bundles.

If this cell fails with a missing API key error, set `MISTRAL_API_KEY` in `.env` or in the notebook kernel environment and rerun from the setup cell.


In [7]:
if not os.environ.get("MISTRAL_API_KEY", "").strip():
    raise RuntimeError(
        "Missing MISTRAL_API_KEY. Add it to .env or the notebook kernel environment, "
        "then rerun the setup cell."
    )

live_env = parse_url(PDF_URL, config=live_config)

written = write_envelope(
    live_env,
    bundle_dir,
    Bundles(notices=True, tables=True, document_index=True, schema=True),
)

validated = validate_envelope_json(written["envelope"])

summary = {
    "run_name": validated.source.run_name,
    "mistral_model": validated.mistral.model,
    "mistral_replay": validated.mistral.request_options.get("replay"),
    "page_count": validated.stats.page_count,
    "notice_count": validated.stats.notice_count,
    "table_count": validated.stats.table_count,
    "raw_json_path": display_path(Path(validated.mistral.raw_json_path)),
    "written_bundles": sorted(written),
}

bundle_lines = "\n".join(
    f"- `{bundle_name}`: `{display_path(bundle_path)}`"
    for bundle_name, bundle_path in sorted(written.items())
)
report = f"""## Live Mistral PDF Test Complete

- Source URL: `{PDF_URL}`
- Run name: `{summary['run_name']}`
- Mistral model: `{summary['mistral_model']}`
- Replay mode: `{summary['mistral_replay']}`
- Page count: `{summary['page_count']}`
- Notice count: `{summary['notice_count']}`
- Table count: `{summary['table_count']}`
- Raw Mistral JSON cache: `{summary['raw_json_path']}`

Written bundles:
{bundle_lines}
"""

show_markdown(report)
summary


## Live Mistral PDF Test Complete

- Source URL: `https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf`
- Run name: `gazette_2009-12-11_103`
- Mistral model: `mistral-ocr-latest`
- Replay mode: `False`
- Page count: `64`
- Notice count: `324`
- Table count: `32`
- Raw Mistral JSON cache: `examples/_live_mistral_pdf_test/stage/gazette_2009-12-11_103.raw.json`

Written bundles:
- `document_index`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_index.json`
- `envelope`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_envelope.json`
- `joined_markdown`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_joined.md`
- `notices`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_notices.json`
- `raw_mistral_json`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103.raw.json`
- `schema`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_schema.json`
- `source_metadata`: `ex

## Live Mistral PDF Test Complete

- Source URL: `https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf`
- Run name: `gazette_2009-12-11_103`
- Mistral model: `mistral-ocr-latest`
- Replay mode: `False`
- Page count: `64`
- Notice count: `324`
- Table count: `32`
- Raw Mistral JSON cache: `examples/_live_mistral_pdf_test/stage/gazette_2009-12-11_103.raw.json`

Written bundles:
- `document_index`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_index.json`
- `envelope`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_envelope.json`
- `joined_markdown`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_joined.md`
- `notices`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_notices.json`
- `raw_mistral_json`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103.raw.json`
- `schema`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_schema.json`
- `source_metadata`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_source.json`
- `tables`: `examples/_live_mistral_pdf_test/bundles/gazette_2009-12-11_103_tables.json`


{'run_name': 'gazette_2009-12-11_103',
 'mistral_model': 'mistral-ocr-latest',
 'mistral_replay': False,
 'page_count': 64,
 'notice_count': 324,
 'table_count': 32,
 'raw_json_path': 'examples/_live_mistral_pdf_test/stage/gazette_2009-12-11_103.raw.json',
 'written_bundles': ['document_index',
  'envelope',
  'joined_markdown',
  'notices',
  'raw_mistral_json',
  'schema',
  'source_metadata',
  'tables']}

In [8]:
import json

raw_payload = json.loads(Path(validated.mistral.raw_json_path).read_text(encoding="utf-8"))
joined_preview = Path(written["joined_markdown"]).read_text(encoding="utf-8").strip()
notices_payload = json.loads(Path(written["notices"]).read_text(encoding="utf-8"))
first_notice = notices_payload[0] if notices_payload else None
first_table = first_notice["tables"][0] if first_notice and first_notice["tables"] else None
joined_excerpt = joined_preview[:2500]

if first_notice is None:
    notice_summary = "No notices were parsed from this OCR response."
else:
    notice_summary = (
        f"- Notice ID: `{first_notice['notice_id']}`\n"
        f"- Notice number: `{first_notice.get('notice_no')}`\n"
        f"- Title lines: `{'; '.join(first_notice.get('title_lines', []))}`\n"
        f"- Dates found: `{', '.join(first_notice.get('dates_found', []))}`\n"
        f"- Confidence band: `{first_notice['confidence_scores']['band']}`"
    )

if first_table is None:
    table_summary = "No tables were parsed from the first notice."
else:
    table_summary = (
        f"- Headers: `{', '.join(first_table['headers'])}`\n"
        f"- First row: `{first_table['records'][0] if first_table['records'] else {}}`"
    )

preview = f"""## Live OCR Output Preview

Raw Mistral cache file: `{display_path(Path(validated.mistral.raw_json_path))}`

Raw Mistral top-level keys: `{', '.join(sorted(raw_payload.keys()))}`

First parsed notice:

{notice_summary}

First parsed table:

{table_summary}

Joined markdown excerpt:

```text
{joined_excerpt}
```
"""

show_markdown(preview)

## Live OCR Output Preview

Raw Mistral cache file: `examples/_live_mistral_pdf_test/stage/gazette_2009-12-11_103.raw.json`

Raw Mistral top-level keys: `document_annotation, model, pages, usage_info`

First parsed notice:

- Notice ID: `gazette_2009-12-11_103:page:1:1:13174`
- Notice number: `13174`
- Title lines: `OFFICE OF THE DEPUTY PRIME MINISTER AND MINISTRY OF FINANCE`
- Dates found: `16th November, 2009`
- Confidence band: `high`

First parsed table:

- Headers: `Receipts, Printed Est. 2009/2010 KShs., Actual Receipts 31/10/09 KShs.`
- First row: `{'Actual Receipts 31/10/09 KShs.': '149,340,890,683.80', 'Printed Est. 2009/2010 KShs.': '512,238,070,771.00', 'Receipts': 'Revenue from Taxation'}`

Joined markdown excerpt:

```text
---

# Document: document_index=0

---

## Index 0

![img-0.jpeg](img-0.jpeg)

# THE KENYA GAZETTE

Published by Authority of the Republic of Kenya

(Registered as a Newspaper at the G.P.O.)

Vol. CXI—No. 103

NAIROBI, 11th December, 2009

Price Sh. 50



## Live OCR Output Preview

Raw Mistral cache file: `examples/_live_mistral_pdf_test/stage/gazette_2009-12-11_103.raw.json`

Raw Mistral top-level keys: `document_annotation, model, pages, usage_info`

First parsed notice:

- Notice ID: `gazette_2009-12-11_103:page:1:1:13174`
- Notice number: `13174`
- Title lines: `OFFICE OF THE DEPUTY PRIME MINISTER AND MINISTRY OF FINANCE`
- Dates found: `16th November, 2009`
- Confidence band: `high`

First parsed table:

- Headers: `Receipts, Printed Est. 2009/2010 KShs., Actual Receipts 31/10/09 KShs.`
- First row: `{'Actual Receipts 31/10/09 KShs.': '149,340,890,683.80', 'Printed Est. 2009/2010 KShs.': '512,238,070,771.00', 'Receipts': 'Revenue from Taxation'}`

Joined markdown excerpt:

```text
---

# Document: document_index=0

---

## Index 0

![img-0.jpeg](img-0.jpeg)

# THE KENYA GAZETTE

Published by Authority of the Republic of Kenya

(Registered as a Newspaper at the G.P.O.)

Vol. CXI—No. 103

NAIROBI, 11th December, 2009

Price Sh. 50

## GAZETTE NOTICES

|  Office of the Deputy Prime Minister and Ministry of Finance—Statement of Actual Revenue and Net Exchequer Issues as at 31st October, 2009 | 3508  |
| --- | --- |
|  The Geologists Registration Act—Appointment | 3508–3509  |
|  The National Museums and Heritage Act—Appointment | 3508  |
|  The Kenya National Library Service Board Act—Appointment | 3509  |
|  The Labour Relations Act—Collection of Agency Fees | 3509  |
|  The Criminal Procedure Code—Appointment | 3509  |
|  The Children Act—Appointment | 350—  |
|  The Medical Practitioners and Dentists Act—Approved Institutions | 3510  |
|  The Registration of Titles Act—Issue of Provisional Certificate | 3510–3511  |
|  The Registered Land Act—Issue of New Land Title Deeds, etc. | 3511–3520  |
|  Probate and Administration | 3521–3551  |
|  The Physical Planning Act—Completion of Part Development Plans | 3551–3553  |
|  The Co-operative Societies Act—Appointment of Liquidator, etc. | 3553  |
|  The Customs and Excise Act—Approved Manufacturers and Producers | 3553  |
|  The Transfer of Business Act | 3553  |
|  The Kenya Communications Act—Application for Licences | 3553–3554  |
|  The Companies Act—Winding-up | 3554  |
|  The Environmental Management and Co-ordination Act—Environmental Impact Assessment Study Report | 3554–3559  |

## CONTENTS

|  PAGE | GAZETTE NOTICES—(Contd.) | PAGE  |
| --- | --- | --- |
|  Local Government Notices |  | 3560–3562  |
|  The Bankruptcy Act—Receiving Order and Creditors’ Meeting |  | 3562  |
|  The Kenya Power and Lighting Company—Fuel Cost Charges, etc. |  | 3563  |
|  Closure of Roads |  | 3563–3564  |
|  Disposal of Uncollected Goods |  | 3564  |
|  Loss of Policies |  | 3564–3566  |
|  Change of Names |  | 3566–3567  |

## SUPPLEMENT No. 81

Legislative Supplement

|  LEGAL NOTICE NO. | PAGE  |
| --- | --- |
|  174—The Immigration Regulations | 725  |
|  175—The Immigration (Amendment) Regulations, 2009 | 727  |
|  176—The Kenya National Examination Council (Kenya Certificate of Secondary Education Examinations) Rules, 2009 | 728  |
|  177—The Registered Land (Application) (No. 15) Order, 2009 | 743  |
|  178—The Registered Land (Application) (No. 16) Order, 2009 | 743  |
|  179—The Registered L
```


## How To Judge The Test

The live test worked if the run cell shows:

- `Live Mistral PDF Test Complete`
- `Replay mode: False`
- a raw JSON cache under `examples/_live_mistral_pdf_test/stage`
- bundle files under `examples/_live_mistral_pdf_test/bundles`
- a joined markdown excerpt from the PDF

If Mistral returns OCR but the parser finds few or zero notices, that is a parser/data-quality issue, not a Mistral connectivity failure.

## Optional: Change The PDF URL

To test another public PDF, change `PDF_URL` in the configuration cell and rerun the configuration, live Mistral, and preview cells.

The URL must be publicly reachable by Mistral. Private local files are not sent by this package yet.


In [ ]:
show_markdown(
    "## Current PDF URL\n\n"
    f"`{PDF_URL}`\n\n"
    "Edit `PDF_URL` in the configuration cell above to test a different public PDF."
)


## Local PDF Files

This package does not yet upload local PDFs to Mistral. The live path currently uses Mistral's `document_url` mode, so the PDF must be reachable by URL.

If you need to test local PDF upload next, that is a separate feature: add an upload flow, then wire `parse_file(..., allow_live_mistral=True)` through it.


In [ ]:
show_markdown(
    "## Local PDF Upload Not Implemented\n\n"
    "This notebook tests live Mistral on a public PDF URL. "
    "Local file upload is not implemented in the package yet."
)
